# 03. Packages and environments

**Tier:** Intermediate. **Why you'd use this:** You need `pylearn.check` in an exercise. That name comes from a package installed into your environment, not from a loose file you happen to have open.

**After this lesson you can:** tell a package import from a local file; explain what `uv add` changes; find a package name on an imported module.
About 25 minutes. **Uses:** [Imports](01-imports-lesson.ipynb)


## Quick review

1. When does a file's `__main__` block run?
2. Why can a notebook not show that difference alone?

<details><summary>Answers</summary>

When the file is the program entry. The notebook is already named `"__main__"`.
</details>


## The idea

A package is an importable tree of modules, usually installed into an environment. `import pylearn` loads that package from the environment on `sys.path`. `uv add` records a dependency in `pyproject.toml`, locks versions in `uv.lock`, and installs into `.venv`. That is different from copying a `.py` file beside your notebook.

<details><summary>Explain more</summary>

`pylearn.__name__` is the package name. Tools and libraries live in the environment so every project gets a reproducible set of packages.
</details>


## See it work

Import the course package and inspect its name.


In [1]:
import pylearn

# 1. Installed packages expose a stable import name.
print(pylearn.__name__)
print(hasattr(pylearn, "check"))

pylearn
True


## Check yourself

Answer these before you run the cell.

1. Does `uv add` only download files into the current folder?
2. Where does `import pylearn` look first for a match?


In [2]:
import sys

import pylearn

print(pylearn.__name__)
print(any("site-packages" in p or ".venv" in p for p in sys.path) or sys.path[0] == "")

pylearn
True


<details><summary>Answers</summary>

No. It updates project metadata, the lock file, and the environment. Along `sys.path`, which includes the environment's site-packages.
</details>


## Why it works this way

Import still searches `sys.path`. An environment puts its `site-packages` on that path so `import pylearn` finds the installed distribution. `uv add` edits `pyproject.toml` to declare the dependency, writes exact versions into `uv.lock` for reproducibility, and syncs wheels into `.venv`. Deep dive: [The import system](../../internals/09-imports/01-import-system-lesson.ipynb).

A loose `.py` file beside you is a module, not an installed package, unless you manipulate path by hand. Projects prefer declared dependencies so another machine can recreate the same imports. `python -m` runs a module as `__main__` with the package context on `sys.path`, which is why package entry points often use that form.

Cost of adding a package is download plus install once per environment, then ordinary imports. The design reason is isolation: each project can pin versions without fighting the system Python.

Evidence: `pylearn.__name__` prints `pylearn`, and `check` is an attribute of that package after install.

Lock files matter when two machines must see the same versions. Without a lock, "works on my machine" returns. Environments matter when two projects need different versions of the same library. The import system does not invent isolation by itself; the environment layout does.  Reading `pyproject.toml` shows declared intent. Reading `uv.lock` shows exact resolution. Reading `sys.path` shows where import will search. Those three views answer most "why can't I import this" questions before you blame the language. If import fails after `uv add`, sync the environment and confirm the active interpreter is the project `.venv`.


In [3]:
import sys

import pylearn

print(pylearn.__name__)
print(hasattr(pylearn, "check"))
print([p for p in sys.path if "site-packages" in p or ".venv" in p][:2])

pylearn
True
['/Users/rickzakharov/dev/learning/python/.venv/lib/python3.14/site-packages']


## When to use it

Use `uv add` for third-party dependencies. Use local modules for your own project code under version control. Use `python -m` to run package entry points. Don't copy library source into the project to "avoid" packaging when a dependency declaration would do. Treat `.venv` as replaceable build output, not as the place you author code.  Commit lockfiles when the project expects reproducible installs.


## Common mistake

Editing code in the environment's `site-packages` copy of a library, then losing the edit on the next sync.


In [4]:
import os

import pylearn

print(os.path.isdir(os.path.dirname(pylearn.__file__)))
print("edit project code, not site-packages")

True
edit project code, not site-packages


Keep your code in the repo. Upgrade dependencies with the package tool when you need upstream changes.


In [5]:
import pylearn

print(pylearn.__name__)
print(callable(pylearn.check))

pylearn
True


## Try it

Run `uv add` help or read `pyproject.toml` in the repo root. Which table lists dependencies, and what file pins exact versions?


## Recap

- Packages are installed into an environment on `sys.path`.
- `uv add` updates `pyproject.toml`, `uv.lock`, and `.venv`.
- Import names come from the distribution, not from your open editor tab.

**Next:** [practice](03-packages-practice.ipynb)
